# Teleoperation with asMagic and mink

In [1]:
from pathlib import Path
import numpy as np
import mujoco
import mujoco.viewer
from loop_rate_limiters import RateLimiter
import mink
from asmagic import ARDataSubscriber
from scipy.spatial.transform import Rotation as R

try:
    ROOT = Path(__file__).resolve().parent   # works in .py scripts
except NameError:
    ROOT = Path.cwd()

In [2]:
_XML = ROOT / "scene.xml"
model = mujoco.MjModel.from_xml_path(_XML.as_posix())

configuration = mink.Configuration(model)
hands = ["right_wrist", "left_wrist"]

tasks = [
    base_orientation_task := mink.FrameTask(
        frame_name="base_link",
        frame_type="body",
        position_cost=0.0,
        orientation_cost=10.0,
    ),
    posture_task := mink.PostureTask(model, cost=10.0),
    com_task := mink.ComTask(cost=200.0),
]

hand_tasks = []
for hand in hands:
    task = mink.FrameTask(
        frame_name=hand,
        frame_type="site",
        position_cost=200.0,
        orientation_cost=0.0,
        lm_damping=2.0,
    )
    hand_tasks.append(task)
tasks.extend(hand_tasks)

head_tasks = []
for head in ["head_link"]:
    task = mink.FrameTask(
        frame_name=head,
        frame_type="body",
        position_cost=0.0,
        orientation_cost=10.0,
        lm_damping=5.0,
    )
    head_tasks.append(task)
tasks.extend(head_tasks)

com_mid = model.body("com_target").mocapid[0]
hands_mid = [model.body(f"{hand}_target").mocapid[0] for hand in hands]
head_mid = model.body("head_target").mocapid[0]

model = configuration.model
data = configuration.data
solver = "daqp"

# TODO: modify the IP as shown in your asMagic APP
left_phone_subscriber = ARDataSubscriber("192.168.31.84")
right_phone_subscriber = ARDataSubscriber("192.168.31.84")

# Swap iPhone axes for teleop: exchange x and z (keep y).
phone_pose_conv_mat = np.array(
    [
        [1, 0, 0, 0], 
        [0, 1, 0, 0],  
        [0, 0, 1, 0],  
        [0, 0, 0, 1],
    ],
    dtype=np.float32,
)
def convert_pose(pose: np.ndarray, conversion_matrix: np.ndarray) -> np.ndarray:
    """
    Convert a pose by applying a conversion matrix.

    Args:
        pose (np.ndarray): The pose to convert, expected to be a 7-element array (x, y, z, qx, qy, qz, qw).
        conversion_matrix (np.ndarray): The conversion matrix to apply.

    Returns:
        np.ndarray: The converted pose, which is also a 7-element array (x, y, z, qx, qy, qz, qw).
    """

    mat = np.eye(4, dtype=np.float32)
    mat[:3, :3] = R.from_quat(pose[3:]).as_matrix()
    mat[:3, 3] = pose[:3]

    converted_mat = conversion_matrix @ mat
    converted_pose = np.zeros_like(pose, dtype=np.float32)
    converted_pose[:3] = converted_mat[:3, 3]
    converted_pose[3:] = R.from_matrix(converted_mat[:3, :3]).as_quat()
    return converted_pose

def get_delta(init, curr):
    # np.roll moves qw from end to start for mink (xyzw -> wxyz)
    T_0 = mink.SE3.from_rotation_and_translation(
        mink.SO3(wxyz=np.roll(init[3:], 1)), init[:3]
    )
    T_t = mink.SE3.from_rotation_and_translation(
        mink.SO3(wxyz=np.roll(curr[3:], 1)), curr[:3]
    )
    return T_0.inverse() @ T_t

In [3]:
with mujoco.viewer.launch_passive(
    model=model, data=data, show_left_ui=False, show_right_ui=False
) as viewer:
    mujoco.mjv_defaultFreeCamera(model, viewer.cam)

    configuration.update_from_keyframe("fix-base")
    posture_task.set_target_from_configuration(configuration)
    base_orientation_task.set_target_from_configuration(configuration)

    # Initialize mocap bodies at their respective sites.
    for hand in hands:
        mink.move_mocap_to_frame(model, data, f"{hand}_target", hand, "site")
    mink.move_mocap_to_frame(model, data, "head_target", "head_link", "body")
    data.mocap_pos[com_mid] = data.subtree_com[1]
    
    # Store initial Robot Hand transforms
    # right_palm is index 0, left_palm is index 1
    T_right_hand_init = mink.SE3.from_mocap_id(data, hands_mid[0])
    T_left_hand_init = mink.SE3.from_mocap_id(data, hands_mid[1])

    viewer.sync()
    
    # Wait for phone streams
    while True:
        try:
            left_raw = left_phone_subscriber.get().global_pose
            init_left_phone_pose = convert_pose(np.array(left_raw, dtype=np.float32), phone_pose_conv_mat)
            right_raw = right_phone_subscriber.get().global_pose
            init_right_phone_pose = convert_pose(np.array(right_raw, dtype=np.float32), phone_pose_conv_mat)
            break
        except Exception as e:
            print("Waiting for phone poses...", e)

    print("Teleop Initialized. Left phone controls Right hand.")

    rate = RateLimiter(frequency=200.0, warn=False)
    
    while viewer.is_running():
        com_task.set_target(data.mocap_pos[com_mid])

        try:
            # 1. Get current phone poses
            left_curr_raw = left_phone_subscriber.get().global_pose
            right_curr_raw = right_phone_subscriber.get().global_pose
            
            left_curr = convert_pose(np.array(left_curr_raw, dtype=np.float32), phone_pose_conv_mat)
            right_curr = convert_pose(np.array(right_curr_raw, dtype=np.float32), phone_pose_conv_mat)

            # 2. Function to calculate relative movement (Delta)
            delta_left_phone = get_delta(init_left_phone_pose, left_curr)
            delta_right_phone = get_delta(init_right_phone_pose, right_curr)

            # 3. Apply Delta to Hand Tasks with Mirroring:
            # User Left Phone -> Robot Right Hand (hand_tasks[0])
            # User Right Phone -> Robot Left Hand (hand_tasks[1])
            hand_tasks[0].set_target(T_right_hand_init @ delta_left_phone)
            hand_tasks[1].set_target(T_left_hand_init @ delta_right_phone)

        except Exception as e:
            pass # Keep previous target if frame is dropped

        head_pitch = 0.0
        head_yaw = 0.0
        head_tasks[0].set_target(mink.SE3.from_rotation(mink.SO3.from_rpy_radians(roll=0.0, pitch=head_pitch, yaw=head_yaw)))

        # IK Solve
        vel = mink.solve_ik(configuration, tasks, rate.dt, solver, 1e-1)
        configuration.integrate_inplace(vel, rate.dt)
        mujoco.mj_camlight(model, data)

        # Update mocap visualizers
        for hand in hands:
            mink.move_mocap_to_frame(model, data, f"{hand}_target", hand, "site")

        viewer.sync()
        rate.sleep()

Waiting for phone poses... 'NoneType' object has no attribute 'global_pose'
Waiting for phone poses... 'NoneType' object has no attribute 'global_pose'
Waiting for phone poses... 'NoneType' object has no attribute 'global_pose'
Waiting for phone poses... 'NoneType' object has no attribute 'global_pose'
Waiting for phone poses... 'NoneType' object has no attribute 'global_pose'
Waiting for phone poses... 'NoneType' object has no attribute 'global_pose'
Waiting for phone poses... 'NoneType' object has no attribute 'global_pose'
Waiting for phone poses... 'NoneType' object has no attribute 'global_pose'
Waiting for phone poses... 'NoneType' object has no attribute 'global_pose'
Waiting for phone poses... 'NoneType' object has no attribute 'global_pose'
Teleop Initialized. Left phone controls Right hand.
